In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
import streamlit as st
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:


# --- CONFIG ---
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 10

# --- LOAD DATA ---
def load_data(img_dir, mask_dir):
    images, masks = [], []
    for filename in os.listdir(img_dir):
        img_path = os.path.join(img_dir, filename)
        mask_path = os.path.join(mask_dir, filename)
        if os.path.exists(mask_path):
            img = cv2.imread(img_path, 0)
            mask = cv2.imread(mask_path, 0)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE)) /255.0
            images.append(img[..., np.newaxis])
            masks.append((mask > 127).astype(np.float32)[..., np.newaxis])
    return np.array(images), np.array(masks)


In [5]:

# --- U-NET MODEL ---
def unet_model(input_size=(256, 256, 1)):
    inputs = tf.keras.layers.Input(input_size)

    def conv_block(x, filters):
        x = tf.keras.layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        return x

    def encoder_block(x, filters):
        f = conv_block(x, filters)
        p = tf.keras.layers.MaxPooling2D()(f)
        return f, p

    def decoder_block(x, skip, filters):
        x = tf.keras.layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
        x = tf.keras.layers.concatenate([x, skip])
        return conv_block(x, filters)

    f1, p1 = encoder_block(inputs, 64)
    f2, p2 = encoder_block(p1, 128)
    f3, p3 = encoder_block(p2, 256)
    f4, p4 = encoder_block(p3, 512)

    bottleneck = conv_block(p4, 1024)

    d1 = decoder_block(bottleneck, f4, 512)
    d2 = decoder_block(d1, f3, 256)
    d3 = decoder_block(d2, f2, 128)
    d4 = decoder_block(d3, f1, 64)

    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)


In [ ]:

# --- STREAMLIT UI ---
st.title("🧠 Train U-Net Model for Brain Tumor Segmentation")

image_dir = st.text_input("Enter path to MRI images folder", "")
mask_dir = st.text_input("Enter path to Mask images folder", "")
model_save_path = st.text_input("Model Save Path", "unet_brain_segmentation.h5")

if st.button("Train U-Net Model"):
    if not os.path.exists(image_dir) or not os.path.exists(mask_dir):
        st.error("❌ Please enter valid paths for image and mask folders.")
    else:
        st.info("📦 Loading data...")
        images, masks = load_data(image_dir, mask_dir)
        if len(images) == 0:
            st.error("❌ No valid image-mask pairs found.")
        else:
            st.success(f"✅ Loaded {len(images)} samples.")
            x_train, x_val, y_train, y_val = train_test_split(images, masks, test_size=0.2, random_state=42)

            st.info("🛠️ Building and training U-Net...")
            model = unet_model()
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

            history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                                epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

            st.success(f"✅ Model trained. Saving to `{model_save_path}`")
            model.save(model_save_path)

            # Plotting metrics
            st.subheader("📈 Training Metrics")
            fig, axs = plt.subplots(1, 2, figsize=(10, 4))
            axs[0].plot(history.history['accuracy'], label='Train Accuracy')
            axs[0].plot(history.history['val_accuracy'], label='Val Accuracy')
            axs[0].set_title("Accuracy")
            axs[0].legend()

            axs[1].plot(history.history['loss'], label='Train Loss')
            axs[1].plot(history.history['val_loss'], label='Val Loss')
            axs[1].set_title("Loss")
            axs[1].legend()

            st.pyplot(fig)


In [10]:
!streamlit run c:\Users\plali\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


^C
